# Almonds (Shelled Basis) — Forecasting Demand & Supply

Time-series analysis and forecasting of almond **production, consumption, exports and
imports** across the major producing and consuming regions, using the USDA
**Production, Supply & Distribution (PSD)** database.

**Data.** `almonds_raw.csv` — the `Almonds, Shelled Basis` slice of USDA PSD
(7,960 rows, 40 countries, market years 1960–2020, metric tonnes).
Source: [USDA FAS PSD Online](https://apps.fas.usda.gov/psdonline/), distributed on
Kaggle as `psd_alldata.csv`.

**The balance sheet.** Supply and demand are two sides of one identity:

    Total Supply       = Beginning Stocks + Production + Imports
    Total Distribution = Domestic Consumption + Exports + Ending Stocks

---

### Terminology

**Market year (`MY`).** The 12-month *marketing* cycle for a commodity, which does not
align with the calendar year — it begins at harvest. For almonds the US market year runs
roughly **1 August – 31 July**, so `MY2000` spans August 2000 to July 2001. The cycle
differs by country and by commodity, which is why USDA carries it as its own field
rather than using calendar dates. Throughout this notebook `MY2020` means "market year
2020", and it is the time axis for every model.

**Shelled basis.** Quantities are expressed as shelled kernel weight, so they are
comparable across countries regardless of whether the nuts are traded in-shell.

**MT.** Metric tonnes — the unit of every value in this dataset.

---

## Structure

| § | Section |
|---|---|
| 1 | Data audit — three structural traps in PSD |
| 2 | Geographic view — where supply and demand actually are |
| 3 | Modelling series and feature engineering |
| 4 | **The model bake-off** — 22 models in three tiers |
| 5 | Why the advanced models lose — the extrapolation ceiling |
| 6 | VAR and the coherence problem |
| 7 | Feature importance |
| 8 | Forecasts to 2025 |
| 9 | Findings, limitations, and proposed fixes |

## 0. Setup

In [ ]:
import os, sys, glob, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# PyTorch powers the LSTM/GRU in Tier 3. If it is unavailable the bake-off simply
# runs without them rather than failing.
#
# Windows note: `pip install torch` (latest) can abort with WinError 206 when
# site-packages sits under a long path — the Microsoft Store Python is a common
# case. torch's bundled licence tree then exceeds the 260-character limit, leaving a
# half-extracted package with no RECORD file that pip cannot uninstall. Installing
# torch==2.5.1, whose licence tree is shallower, avoids it.
try:
    import torch
    import torch.nn as nn
    torch.set_num_threads(2)
    HAS_TORCH = True
except Exception as _e:
    HAS_TORCH = False
    print('PyTorch unavailable — LSTM/GRU will be skipped:', _e)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 6)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
pio.renderers.default = 'notebook'
RNG = 0
print('setup complete | torch:', torch.__version__ if HAS_TORCH else 'n/a')

---
## 1. Data audit — three structural traps

An earlier version of this analysis produced a flat, unmodellable series. Three
properties of PSD are responsible, and all three are easy to miss.

In [ ]:
# Locate almonds_raw.csv across local / Kaggle / Colab.
CSV = None
_candidates = ['almonds_raw.csv', '../input/almonds_raw.csv']
_candidates += sorted(glob.glob('/kaggle/input/*/almonds_raw.csv'))
_candidates += sorted(glob.glob('/kaggle/input/*/*/almonds_raw.csv'))

for _c in _candidates:
    if os.path.exists(_c):
        CSV = _c
        break

if CSV is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CSV = '/content/drive/MyDrive/almonds_raw.csv'
    except ImportError:
        raise FileNotFoundError(
            'almonds_raw.csv not found. Local: put it beside the notebook. '
            'Kaggle: add it as a Dataset (it will appear under /kaggle/input/...). '
            'Colab: copy it to your Drive root.')

print('loading:', CSV)
raw = pd.read_csv(CSV)
print(f'{raw.shape[0]:,} rows x {raw.shape[1]} cols | {raw.Country_Name.nunique()} countries | '
      f'MY{raw.Market_Year.min()}-{raw.Market_Year.max()} | missing {raw.isnull().sum().sum()}')
raw.head()

### Trap 1 — `Market_Year` is the time axis, not `Calendar_Year` / `Month`

`Calendar_Year` and `Month` record when USDA **published** an estimate, not the period
measured.

In [ ]:
bad = pd.to_datetime(raw.Calendar_Year.astype(str) + '-' + raw.Month.astype(str) + '-01',
                     errors='coerce')
print('Calendar_Year+Month -> distinct timestamps:', bad.nunique(),
      '| rows lost to NaT:', f'{bad.isna().sum():,}')
print('   Month values:', sorted(raw.Month.unique()), '  <- 0 is not a month')
print('Market_Year         -> distinct years     :', raw.Market_Year.nunique())

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
bad.dt.year.value_counts().sort_index().plot(kind='bar', ax=ax[0], color='#c44')
ax[0].set_title('Calendar_Year — 17 usable points'); ax[0].set_xlabel('')
raw.Market_Year.value_counts().sort_index().plot(ax=ax[1], color='#2a7')
ax[1].set_title('Market_Year — 61 points'); ax[1].set_xlabel('market year')
plt.tight_layout(); plt.show()

### Trap 2 — the data is **annual**

One observation per country per year. This single fact rules out a large part of the
standard time-series toolkit, and it is worth being explicit about which parts:

| Method | Status here | Why |
|---|---|---|
| **Seasonal Naive** | **not applicable** | Needs a within-year cycle. With annual data it collapses into plain Naive — the same model under a different name |
| **SARIMA** (seasonal terms) | **not applicable** | The seasonal orders `(P,D,Q,s)` have no `s` to estimate. Non-seasonal ARIMA is the correct member of that family |
| `seasonal_decompose` | **not applicable** | Requires ≥2 full seasonal periods |
| **Prophet** | **excluded** | Its strengths — holiday effects, multiple seasonality, irregular/missing dates, changepoints on long daily series — are all irrelevant to 20–61 annual points. It would reduce to a piecewise-linear trend, which the models below already cover |

Plain Naive, ARIMA and ETS with a **trend but no seasonal component** are the correct
members of these families for this dataset, and those are what the bake-off uses.

### Trap 3 — Spain and Italy become **placeholder zeros** from MY2001

USDA folds them into a single `European Union` line. Their country rows persist filled
with `0.0` — read literally, the world's second-largest producer collapses to nothing.

In [ ]:
panel = raw.pivot_table(index=['Country_Name', 'Market_Year'],
                        columns='Attribute_Description', values='Value')
panel.columns.name = None

fig, ax = plt.subplots(figsize=(13, 4.2))
for c in ['Spain', 'Italy', 'European Union']:
    s = panel.loc[c, 'Production'].sort_index()
    ax.plot(s.index, s.values / 1000, marker='o', ms=3, label=c)
ax.axvline(2000.5, color='k', ls='--', lw=1)
ax.annotate('EU aggregate begins (MY2001)\nSpain/Italy rows -> 0',
            xy=(2002, ax.get_ylim()[1] * .65), fontsize=9)
ax.set_title('The Spain / Italy reporting break'); ax.set_ylabel('1000 MT')
ax.set_xlabel('market year'); ax.legend(); plt.show()

for c in ['Spain', 'Italy']:
    post = panel.loc[c, 'Production']; post = post[post.index > 2000]
    print(f'{c}: {(post == 0).sum()} of {len(post)} years 2001-2020 are exactly 0.0')

for c in ['Spain', 'Italy']:
    m = (panel.index.get_level_values(0) == c) & (panel.index.get_level_values(1) > 2000)
    panel.loc[m] = np.nan
panel = panel.dropna(how='all')
print('\npanel:', panel.shape[0], 'country-year rows after masking')

### Validating the reshape against the PSD accounting identities

PSD is a balance sheet, so both identities must hold **exactly**. A free correctness
check on the pivot.

In [ ]:
sup = panel['Beginning Stocks'] + panel['Production'] + panel['Imports']
dis = panel['Domestic Consumption'] + panel['Exports'] + panel['Ending Stocks']
print(f"max |Total Supply identity error|       : {(sup - panel['Total Supply']).abs().max():,.1f} MT")
print(f"max |Total Distribution identity error| : {(dis - panel['Total Distribution']).abs().max():,.1f} MT")
gap = (panel['Total Supply'] - panel['Total Distribution']).abs()
print(f"rows where supply != distribution       : {(gap > 1).sum()} of {len(panel)} "
      f"(max {gap.max():,.0f} MT — PSD statistical residual)")

---
## 2. Geographic view

In [ ]:
ISO3 = {'Afghanistan': 'AFG', 'Algeria': 'DZA', 'Argentina': 'ARG', 'Australia': 'AUS',
        'Brazil': 'BRA', 'Canada': 'CAN', 'Chile': 'CHL', 'China': 'CHN',
        'Colombia': 'COL', 'Greece': 'GRC', 'Hong Kong': 'HKG', 'India': 'IND',
        'Indonesia': 'IDN', 'Iran': 'IRN', 'Israel': 'ISR', 'Italy': 'ITA',
        'Japan': 'JPN', 'Jordan': 'JOR', 'Kazakhstan': 'KAZ', 'Korea, South': 'KOR',
        'Malaysia': 'MYS', 'Mexico': 'MEX', 'Morocco': 'MAR', 'New Zealand': 'NZL',
        'Norway': 'NOR', 'Pakistan': 'PAK', 'Portugal': 'PRT', 'Russia': 'RUS',
        'Saudi Arabia': 'SAU', 'South Africa': 'ZAF', 'Spain': 'ESP',
        'Switzerland': 'CHE', 'Taiwan': 'TWN', 'Thailand': 'THA', 'Tunisia': 'TUN',
        'Turkey': 'TUR', 'United Arab Emirates': 'ARE', 'United States': 'USA',
        'Vietnam': 'VNM'}
EU_MEMBERS = ['AUT', 'BEL', 'BGR', 'HRV', 'CYP', 'CZE', 'DNK', 'EST', 'FIN', 'FRA',
              'DEU', 'GRC', 'HUN', 'IRL', 'ITA', 'LVA', 'LTU', 'LUX', 'MLT', 'NLD',
              'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'ESP', 'SWE']


def to_map_frame(series, value_name='value'):
    """Country series -> one row per ISO3 code.

    The EU is a bloc, not a country, so its value is painted across member states.
    Members PSD *also* reports individually (e.g. Greece) keep their own value,
    otherwise the bloc figure would overwrite and double-count them.
    """
    own = {ISO3[c]: (v, c) for c, v in series.items() if c in ISO3 and pd.notna(v)}
    rows = []
    if 'European Union' in series.index and pd.notna(series['European Union']):
        for iso in EU_MEMBERS:
            if iso not in own:
                rows.append({'iso': iso, value_name: series['European Union'],
                             'label': 'European Union (bloc)'})
    for iso, (v, c) in own.items():
        rows.append({'iso': iso, value_name: v, 'label': c})
    return pd.DataFrame(rows)


YEAR = 2020
snap = panel.xs(YEAR, level=1)
fig = px.choropleth(to_map_frame(snap['Production'], 'Production'),
                    locations='iso', color='Production', hover_name='label',
                    color_continuous_scale='YlOrBr',
                    title=f'Almond production by country — MY{YEAR} (MT)')
fig.update_layout(height=450, margin=dict(l=0, r=0, t=50, b=0)); fig.show()

In [ ]:
fig = px.choropleth(to_map_frame(snap['Domestic Consumption'], 'Consumption'),
                    locations='iso', color='Consumption', hover_name='label',
                    color_continuous_scale='Greens',
                    title=f'Almond domestic consumption — MY{YEAR} (MT)')
fig.update_layout(height=450, margin=dict(l=0, r=0, t=50, b=0)); fig.show()

net = (snap['Exports'] - snap['Imports']).dropna()
mn = to_map_frame(net, 'NetTrade'); lim = mn.NetTrade.abs().max()
fig = px.choropleth(mn, locations='iso', color='NetTrade', hover_name='label',
                    color_continuous_scale='RdBu', range_color=(-lim, lim),
                    title=f'Net trade position — MY{YEAR} (exports - imports, MT)')
fig.update_layout(height=450, margin=dict(l=0, r=0, t=50, b=0)); fig.show()

In [ ]:
frames = []
for yr in range(2001, 2021):
    try:
        f = to_map_frame(panel.xs(yr, level=1)['Production'], 'Production')
    except KeyError:
        continue
    f['year'] = yr; frames.append(f)
anim = pd.concat(frames)
fig = px.choropleth(anim, locations='iso', color='Production', hover_name='label',
                    animation_frame='year', color_continuous_scale='YlOrBr',
                    range_color=(0, anim.Production.quantile(.995)),
                    title='Almond production over time (press play)')
fig.update_layout(height=500, margin=dict(l=0, r=0, t=50, b=0)); fig.show()

In [ ]:
world = panel.groupby(level=1)[['Production', 'Domestic Consumption']].sum()
us_share = (panel.loc['United States', 'Production'].reindex(world.index)
            / world['Production'] * 100).dropna()
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
world.div(1000).plot(ax=ax[0]); ax[0].set_title('World production vs consumption')
ax[0].set_ylabel('1000 MT'); ax[0].set_xlabel('market year')
us_share.plot(ax=ax[1], color='#b5651d'); ax[1].set_title('US share of world production')
ax[1].set_ylabel('%'); ax[1].set_xlabel('market year'); ax[1].set_ylim(0, 100)
plt.tight_layout(); plt.show()
print(f'US share of world production in {int(us_share.index[-1])}: {us_share.iloc[-1]:.1f}%')

**Supply is concentrated; demand is dispersed.** The US alone is ~80% of world
production and its share has *risen*. Consumption is spread across the EU, India, China
and the Middle East. The market is structurally exposed to conditions in one region —
California — which is the central risk in any almond outlook.

---
## 3. Modelling series and features

In [ ]:
FOCUS = ['United States', 'European Union', 'Australia', 'Spain', 'Italy']
TARGETS = ['Production', 'Domestic Consumption', 'Exports', 'Imports']
ATTRS = ['Production', 'Domestic Consumption', 'Exports', 'Imports',
         'Beginning Stocks', 'Ending Stocks']
N_LAGS = 3


def country_frame(country):
    """All attributes on a gap-free annual index.

    A few market years are absent (1963 for everyone; 1977-1980 for the US). ARIMA and
    lag features need regular spacing, so those are linearly interpolated.
    """
    df = panel.loc[country, ATTRS].dropna(how='all').sort_index()
    return df.reindex(range(int(df.index.min()), int(df.index.max()) + 1)).interpolate('linear')


def get_series(country, attr):
    return country_frame(country)[attr]


def make_xy(country, target, n_lags=N_LAGS):
    """Design matrix. EVERY feature is lagged >=1 — the other attributes for year t
    are unknown when forecasting year t, so using them would be leakage."""
    df = country_frame(country)
    feat = {}
    for col in df.columns:
        for L in range(1, n_lags + 1):
            feat[f'{col} (t-{L})'] = df[col].shift(L)
    feat['time index'] = df.index.to_series().values - int(df.index.min())
    X = pd.DataFrame(feat, index=df.index)
    y = df[target]
    ok = X.notna().all(axis=1) & y.notna()
    return X[ok], y[ok]


for c in FOCUS:
    s = get_series(c, 'Production')
    gaps = sorted(set(s.index) - set(panel.loc[c, 'Production'].dropna().index))
    print(f'{c:16s} n={len(s):3d}  {int(s.index.min())}-{int(s.index.max())}  interpolated: {gaps}')
print()
Xd, yd = make_xy('United States', 'Production')
print(f'US Production design matrix: {Xd.shape[0]} rows x {Xd.shape[1]} features')

In [ ]:
rows = []
for c in FOCUS:
    for t in TARGETS:
        s = get_series(c, t)
        if len(s) < 12 or (s == 0).all():
            continue
        rows.append({'country': c, 'series': t, 'n': len(s),
                     'ADF p (level)': adfuller(s.dropna())[1],
                     'ADF p (1st diff)': adfuller(s.diff().dropna())[1]})
adf_table = pd.DataFrame(rows)
adf_table['stationary at d=1'] = adf_table['ADF p (1st diff)'] <= .05
adf_table

Almost everything is non-stationary in levels. One difference fixes most.

> **Choosing `d`.** Grid-searching `(p,d,q)` jointly on AIC is invalid — differencing
> changes the number of observations the likelihood is computed over, so **AIC is not
> comparable across different `d`**. Doing it anyway drives the search to `d=2` almost
> everywhere (over-differencing). Below, `d` comes from repeated ADF testing, then
> `p`,`q` are gridded by AIC *within* that `d`.

---
## 4. The model bake-off — 22 models in three tiers

### Tier 1 — Baselines (mandatory)
Naive, Drift, Historical mean. *Seasonal Naive is not applicable — see Trap 2.*

### Tier 2 — Statistical
ARIMA, ETS (Holt linear trend), Theta, **VAR**. *SARIMA is not applicable — see Trap 2.*

### Tier 3 — Machine learning & deep learning
LinearRegression, Ridge, Lasso, ElasticNet, RandomForest, ExtraTrees,
GradientBoosting, XGBoost, LightGBM, SVR, KNN, MLP ×2, **LSTM**, **GRU**.
*Prophet excluded — see Trap 2.*

**Every model stays in the results, including the ones that lose.** Removing weak
performers would not reduce bias, it would create it: the case that simple methods beat
complex ones only exists if the complex ones were actually tested and reported.

### Two decisions that make this a fair comparison

**1. Rolling-origin (walk-forward) validation, never `train_test_split`.**
A random split would train on 2019 and test on 1985 — leakage that produces meaningless
scores. Here the model is refit at each origin on the past only, predicting one year
ahead.

**2. Targets are scaled for the scale-sensitive models.**
Almond quantities run to ~10⁶ MT. SVR's default `epsilon`/`C` and neural-net weight
initialisation assume roughly unit-scale targets; without `TransformedTargetRegressor`,
SVR scores 77% MAPE and looks useless — with it, 23%. LightGBM's default
`min_child_samples=20` also exceeds the training row count here and silently returns a
constant. Both are configuration artefacts, not model failures, and correcting them is
required for the comparison to mean anything.

### Why test models that "shouldn't" work here?

The standard advice is to use models built for temporal structure — ETS, ARIMA, VAR —
and not to reach for general-purpose regressors on a time series. That advice is sound,
and the results below confirm it. So why run fifteen models the textbook would have told
us to skip?

**1. A baseline comparison is only evidence if the alternatives were actually tried.**
"Use Naive or Drift to prove your complex models add value" is the right instinct, but
the proof requires the complex models to be *in the table*. Reporting only ETS and ARIMA
would reduce "simple methods win here" from a finding to an assertion.

**2. Negative results carry mechanism, not just a ranking.** The extrapolation ceiling
in §5 — a Random Forest emitting a flat line for eight years while production rises 48% —
only exists because tree models were run. That single plot explains *why* the classical
models win, which is far more useful than knowing *that* they win.

**3. The received wisdom is a heuristic, and heuristics have exceptions worth finding.**
Two turned up here that the rule of thumb would have hidden:

- **LightGBM placed 2nd overall**, ahead of ARIMA. That is hard to square with a blanket
  "tree models don't do time series" — and it only appeared after correcting a default
  (`min_child_samples`) that had been silently crippling it.
- **The LSTM beat a 4x larger MLP on 16 of 20 series.** At 58 training rows, "no deep
  learning" would have been the safe call — and it would have missed a real result about
  architecture substituting for data.

**4. It surfaced a concrete hazard.** Plain `LinearRegression` — a very common choice for
lag-feature forecasting — lands near the bottom with a worst case several times its own
median. That is a practical warning you only get by running it.

**5. The cost was trivial.** The entire bake-off is a few minutes of CPU.

### The honest cost of doing this

Testing many models has a real statistical price: **multiple comparisons**. With 22
models across 20 series, some model wins a series by luck alone, and the per-series
"winner" is therefore a noisy statistic. Three habits keep the exploration honest:

- Judge on the **median across series**, not on win counts.
- **Report every model**, so nothing is selected after the fact.
- Treat a winning model's own MAPE as **optimistically biased**, since it was chosen on
  the same backtest it is reported on.

This is exploratory research — the goal is to understand the problem, not to defend a
prior about which family "should" win. Deciding in advance to report all results,
including the embarrassing ones, is what separates curiosity from cherry-picking.

In [ ]:
def scaled(est):
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), est), transformer=StandardScaler())


class TorchRNN(BaseEstimator, RegressorMixin):
    """LSTM / GRU over the lag matrix, exposed through the sklearn interface.

    Each row of X holds lags 1..N_LAGS of every attribute, so it is reshaped into a
    (timesteps x channels) sequence — oldest lag first — for the recurrent layer.
    """

    def __init__(self, kind='lstm', hidden=24, layers=1, epochs=400, lr=1e-2,
                 n_lags=N_LAGS, patience=50, seed=RNG):
        self.kind, self.hidden, self.layers = kind, hidden, layers
        self.epochs, self.lr, self.n_lags = epochs, lr, n_lags
        self.patience, self.seed = patience, seed

    def _seq(self, Xs):
        Xv = np.asarray(Xs, dtype=np.float32)
        extra, seq = Xv[:, -1:], Xv[:, :-1]
        n, f = seq.shape
        seq = seq.reshape(n, f // self.n_lags, self.n_lags)[:, :, ::-1]
        seq = np.transpose(seq, (0, 2, 1))
        extra_t = np.repeat(extra[:, None, :], self.n_lags, axis=1)
        return np.ascontiguousarray(np.concatenate([seq, extra_t], axis=2), dtype=np.float32)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        self.xs_, self.ys_ = StandardScaler(), StandardScaler()
        Xs = self.xs_.fit_transform(np.asarray(X, float))
        ys = self.ys_.fit_transform(np.asarray(y, float).reshape(-1, 1))
        seq = torch.tensor(self._seq(Xs))
        tgt = torch.tensor(ys, dtype=torch.float32)
        rnn = nn.LSTM if self.kind == 'lstm' else nn.GRU
        self.rnn_ = rnn(seq.shape[2], self.hidden, self.layers, batch_first=True)
        self.head_ = nn.Linear(self.hidden, 1)
        opt = torch.optim.Adam(list(self.rnn_.parameters()) + list(self.head_.parameters()),
                               lr=self.lr)
        lossf, best, bad, snap = nn.MSELoss(), np.inf, 0, None
        for _ in range(self.epochs):
            opt.zero_grad()
            out, _ = self.rnn_(seq)
            loss = lossf(self.head_(out[:, -1, :]), tgt)
            loss.backward(); opt.step()
            l = float(loss)
            if l < best - 1e-7:
                best, bad = l, 0
                snap = ([p.detach().clone() for p in self.rnn_.parameters()],
                        [p.detach().clone() for p in self.head_.parameters()])
            else:
                bad += 1
                if bad >= self.patience:
                    break
        if snap:
            with torch.no_grad():
                for p, q in zip(self.rnn_.parameters(), snap[0]): p.copy_(q)
                for p, q in zip(self.head_.parameters(), snap[1]): p.copy_(q)
        return self

    def predict(self, X):
        seq = torch.tensor(self._seq(self.xs_.transform(np.asarray(X, float))))
        with torch.no_grad():
            out, _ = self.rnn_(seq)
            p = self.head_(out[:, -1, :]).numpy()
        return self.ys_.inverse_transform(p).ravel()


def build_models():
    m = {
        'LinearRegression': scaled(LinearRegression()),
        'Ridge':            scaled(Ridge(alpha=10.0)),
        'Lasso':            scaled(Lasso(alpha=0.05, max_iter=50000)),
        'ElasticNet':       scaled(ElasticNet(alpha=0.05, l1_ratio=.5, max_iter=50000)),
        'RandomForest':     RandomForestRegressor(n_estimators=300, random_state=RNG, n_jobs=-1),
        'ExtraTrees':       ExtraTreesRegressor(n_estimators=300, random_state=RNG, n_jobs=-1),
        'GradientBoosting': GradientBoostingRegressor(random_state=RNG),
        'XGBoost':          XGBRegressor(n_estimators=300, learning_rate=.05, max_depth=3,
                                         verbosity=0, random_state=RNG),
        'LightGBM':         LGBMRegressor(n_estimators=300, learning_rate=.05, max_depth=3,
                                          min_child_samples=2, min_split_gain=0.0,
                                          verbose=-1, random_state=RNG),
        'SVR':              scaled(SVR(C=10.0)),
        'KNN':              scaled(KNeighborsRegressor(n_neighbors=3)),
        'NeuralNet (64-32)':     scaled(MLPRegressor(hidden_layer_sizes=(64, 32),
                                                     max_iter=3000, random_state=RNG)),
        'NeuralNet (128-64-32)': scaled(MLPRegressor(hidden_layer_sizes=(128, 64, 32),
                                                     max_iter=3000, random_state=RNG)),
    }
    if HAS_TORCH:
        m['LSTM'] = TorchRNN(kind='lstm')
        m['GRU'] = TorchRNN(kind='gru')
    return m


SERIES_MODELS = ['Naive', 'Drift', 'Mean', 'ARIMA', 'ETS-Holt', 'Theta']
SYSTEM_MODELS = ['VAR']
ALL_MODELS = list(build_models()) + SERIES_MODELS + SYSTEM_MODELS

TIER = {**{m: 'Tier 1 — baseline' for m in ['Naive', 'Drift', 'Mean']},
        **{m: 'Tier 2 — statistical' for m in ['ARIMA', 'ETS-Holt', 'Theta', 'VAR']}}
for m in build_models():
    TIER[m] = 'Tier 3 — ML / deep learning'
print(len(ALL_MODELS), 'models\n' + ', '.join(ALL_MODELS))

In [ ]:
def choose_d(y, alpha=.05, dmax=2):
    d, s = 0, y.copy()
    while d < dmax:
        s2 = s.dropna()
        if len(s2) < 6:
            break
        try:
            if adfuller(s2)[1] <= alpha:
                break
        except Exception:
            break
        s, d = s.diff(), d + 1
    return d


def arima_order(y, maxp=3, maxq=3):
    d = choose_d(y); best, order = np.inf, (1, d, 0)
    for p, q in itertools.product(range(maxp + 1), range(maxq + 1)):
        if p == 0 and q == 0:
            continue
        try:
            m = ARIMA(y, order=(p, d, q), trend='t' if d == 0 else None).fit()
            if np.isfinite(m.aic) and m.aic < best:
                best, order = m.aic, (p, d, q)
        except Exception:
            continue
    return order


def series_forecast(name, s, order=None, h=1):
    if name == 'Naive':
        return s.iloc[-1]
    if name == 'Mean':
        return s.mean()
    if name == 'Drift':
        return s.iloc[-1] + (s.iloc[-1] - s.iloc[0]) / (len(s) - 1) * h
    if name == 'ARIMA':
        o = order or arima_order(s)
        return float(ARIMA(s, order=o, trend='t' if o[1] == 0 else None).fit().forecast(h).iloc[-1])
    if name == 'ETS-Holt':
        return float(ExponentialSmoothing(s, trend='add').fit().forecast(h).iloc[-1])
    if name == 'Theta':
        return float(ThetaModel(s.values, period=1).fit().forecast(h)[-1])
    raise ValueError(name)


VAR_COLS = ['Production', 'Domestic Consumption', 'Exports', 'Imports']

def var_forecast(country, target, train_end, h=1, maxlags=2):
    """Fit VAR on the four-variable balance system; return the forecast for `target`.

    Fitted on first differences (the levels are non-stationary), then cumulated back.
    """
    df = country_frame(country)[VAR_COLS].iloc[:train_end]
    df = df.loc[:, df.std() > 0]
    if target not in df.columns or len(df) < 12:
        return np.nan
    work = df.diff().dropna()
    ml = min(maxlags, max(1, (len(work) - 2) // (len(work.columns) * 2)))
    res = VAR(work).fit(maxlags=ml)
    steps = res.forecast(work.values[-res.k_ar:], h)
    j = list(work.columns).index(target)
    return df[target].iloc[-1] + steps[:, j].sum()

In [ ]:
N_TEST = 10

def rolling_origin_eval(country, target, n_test=N_TEST):
    X, y = make_xy(country, target)
    if len(y) < 12:
        return None
    n_test = min(n_test, len(y) - 8)
    if n_test < 4:
        return None
    order = arima_order(y.iloc[:len(y) - n_test])   # chosen once, not per origin
    yr_index = list(country_frame(country).index)

    preds = {m: [] for m in ALL_MODELS}
    actual = []
    for t in range(len(y) - n_test, len(y)):
        Xtr, ytr, Xte = X.iloc[:t], y.iloc[:t], X.iloc[t:t + 1]
        actual.append(y.iloc[t])
        for name, mdl in build_models().items():
            try:
                mdl.fit(Xtr, ytr); preds[name].append(float(mdl.predict(Xte)[0]))
            except Exception:
                preds[name].append(np.nan)
        for name in SERIES_MODELS:
            try:
                preds[name].append(float(series_forecast(name, ytr, order)))
            except Exception:
                preds[name].append(np.nan)
        try:
            preds['VAR'].append(float(var_forecast(country, target,
                                                   yr_index.index(y.index[t]))))
        except Exception:
            preds['VAR'].append(np.nan)

    actual = np.asarray(actual, float)
    rows = []
    for name, p in preds.items():
        p = np.asarray(p, float)
        ok = ~np.isnan(p) & (actual != 0)
        if ok.sum() < 3:
            continue
        err = actual[ok] - p[ok]
        rows.append({'country': country, 'series': target, 'model': name,
                     'MAE': np.mean(np.abs(err)), 'RMSE': np.sqrt(np.mean(err ** 2)),
                     'MAPE': np.mean(np.abs(err / actual[ok])) * 100})
    return pd.DataFrame(rows), pd.DataFrame(preds, index=y.index[-n_test:]), y[-n_test:]

In [ ]:
t0 = time.time()
all_scores, all_preds = [], {}
for c in FOCUS:
    for t in TARGETS:
        y = get_series(c, t)
        if len(y) < 12 or (y == 0).all() or (y[-N_TEST:] == 0).any():
            continue
        out = rolling_origin_eval(c, t)
        if out is None:
            continue
        sc, pr, act = out
        all_scores.append(sc); all_preds[(c, t)] = (pr, act)
        print(f'{c:16s} {t:22s} best: {sc.loc[sc.MAPE.idxmin(), "model"]:22s} '
              f'({sc.MAPE.min():5.2f}%)')
scores = pd.concat(all_scores, ignore_index=True)
scores['tier'] = scores.model.map(TIER)
print(f'\ndone in {time.time() - t0:.0f}s — {len(all_preds)} series x {scores.model.nunique()} models')

### 4.1 Overall ranking

In [ ]:
rank = (scores.groupby('model')
        .agg(median_MAPE=('MAPE', 'median'), mean_MAPE=('MAPE', 'mean'),
             worst_MAPE=('MAPE', 'max'), series=('MAPE', 'size'))
        .sort_values('median_MAPE'))
rank['wins'] = (scores.loc[scores.groupby(['country', 'series']).MAPE.idxmin(), 'model']
                .value_counts().reindex(rank.index).fillna(0).astype(int))
rank['tier'] = rank.index.map(TIER)
rank.round(2)

In [ ]:
tier_pal = {'Tier 1 — baseline': '#888', 'Tier 2 — statistical': '#2a7',
            'Tier 3 — ML / deep learning': '#c33'}
order = rank.index.tolist()

fig, ax = plt.subplots(figsize=(12, 8))
sns.boxplot(data=scores, y='model', x='MAPE', order=order, ax=ax,
            palette={m: tier_pal[TIER[m]] for m in order}, fliersize=2)
ax.set_xlim(0, min(120, scores.MAPE.quantile(.99)))
ax.set_xlabel('MAPE % across all series (lower is better)'); ax.set_ylabel('')
ax.set_title('Model bake-off — one-step-ahead rolling-origin validation')
ax.legend(handles=[plt.Line2D([], [], marker='s', ls='', color=v, label=k)
                   for k, v in tier_pal.items()], loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

display(scores.groupby('tier').MAPE.agg(['median', 'mean', 'max']).round(2))

In [ ]:
pivot = scores.pivot_table(index='model', columns=['country', 'series'],
                           values='MAPE').reindex(order)
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn_r', vmin=0, vmax=60,
            cbar_kws={'label': 'MAPE %'}, annot_kws={'size': 7}, ax=ax)
ax.set_title('MAPE by model and series (green = better)'); ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

### 4.2 Did the deep learning models earn their keep?

The recurrent nets are compared against the plain MLPs (same data, same features) and
against the best simple model, to separate *architecture* from *model complexity*.

In [ ]:
nn_models = [m for m in ['LSTM', 'GRU', 'NeuralNet (64-32)', 'NeuralNet (128-64-32)']
             if m in scores.model.values]
simple = ['Naive', 'Drift', 'Mean', 'ARIMA', 'ETS-Holt', 'Theta']

cmp = scores[scores.model.isin(nn_models)].pivot_table(
    index=['country', 'series'], columns='model', values='MAPE')
cmp['best simple'] = (scores[scores.model.isin(simple)]
                      .groupby(['country', 'series']).MAPE.min())
display(cmp.round(2))

for m in nn_models:
    beat = (cmp[m] < cmp['best simple']).sum()
    print(f'{m:24s} median {cmp[m].median():5.2f}%  beats best simple model on '
          f'{beat} of {len(cmp)} series')
if 'LSTM' in cmp and 'NeuralNet (64-32)' in cmp:
    print(f'\nLSTM better than plain MLP on '
          f'{(cmp["LSTM"] < cmp["NeuralNet (64-32)"]).sum()} of {len(cmp)} series')

---
## 5. Why the advanced models lose — the extrapolation ceiling

This is the single most important result, and it is **structural, not a tuning problem**.

A tree-based model predicts by averaging training observations in the same leaf. It
therefore **can never output a value outside the range of its training data**. When a
series trends persistently upward — as almond production does — every tree model is
guaranteed to under-forecast the moment it leaves the training range.

In [ ]:
y_us = get_series('United States', 'Production')
Xt = np.arange(len(y_us)).reshape(-1, 1)
cut = len(y_us) - 8
rf = RandomForestRegressor(n_estimators=300, random_state=RNG).fit(Xt[:cut], y_us.values[:cut])
lr = LinearRegression().fit(Xt[:cut], y_us.values[:cut])

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(y_us.index, y_us.values / 1000, 'k-', lw=1.6, label='actual')
ax.axvline(y_us.index[cut], color='k', ls=':', lw=1)
ax.plot(y_us.index[cut:], rf.predict(Xt[cut:]) / 1000, 'o-', color='#b83',
        label='RandomForest — flat, cannot exceed training max')
ax.plot(y_us.index[cut:], lr.predict(Xt[cut:]) / 1000, 's-', color='#57f',
        label='LinearRegression — extrapolates but under-slopes')
ax.axhline(y_us.values[:cut].max() / 1000, color='#b83', ls='--', lw=1, label='training maximum')
ax.set_title('The extrapolation ceiling — US almond production')
ax.set_ylabel('1000 MT'); ax.set_xlabel('market year'); ax.legend(fontsize=9)
plt.show()

print(f'training maximum       : {y_us.values[:cut].max():>12,.0f} MT')
print(f'actual 2020            : {y_us.values[-1]:>12,.0f} MT')
print(f'RandomForest 2020 pred : {rf.predict(Xt[-1:])[0]:>12,.0f} MT  <- same as its 2013 pred')
print(f'RF prediction range    : {rf.predict(Xt[cut:]).min():,.0f} - {rf.predict(Xt[cut:]).max():,.0f} MT')

The Random Forest emits a **flat line** for eight consecutive years while real
production rises ~48%.

**Why the classical models avoid this.** ARIMA (with `d≥1`), Drift and Holt's method all
model the *change* between periods rather than the level. Differencing removes the
trend, so the trend extrapolates by construction — exactly the mechanism tree models
lack.

**The fix for using ML here:** model **differences or growth rates** rather than levels,
then cumulate predictions back. That gives the tree models a stationary target inside
their training range. This remains the highest-value untested experiment.

---
## 6. VAR and the coherence problem

Every model so far forecasts each series **in isolation**, so nothing forces production
to equal consumption + exports + stock change. VAR fits the four series as one system
where each variable depends on lags of all the others.

The question is whether the joint model buys accuracy, coherence, both, or neither.

In [ ]:
var_scores = scores[scores.model == 'VAR'][['country', 'series', 'MAPE']]
best_other = (scores[scores.model != 'VAR']
              .groupby(['country', 'series']).MAPE.min().rename('best non-VAR'))
cmp_var = var_scores.set_index(['country', 'series']).join(best_other)
cmp_var['VAR better'] = cmp_var.MAPE < cmp_var['best non-VAR']
display(cmp_var.round(2))
print(f"VAR beats the best non-VAR model on {int(cmp_var['VAR better'].sum())} "
      f"of {len(cmp_var)} series")

In [ ]:
# Coherence test: does a VAR system forecast balance better than independent ones?
FUTURE = 5
c = 'United States'
df_us = country_frame(c)[VAR_COLS]
work = df_us.diff().dropna()
res = VAR(work).fit(maxlags=2)
steps = res.forecast(work.values[-res.k_ar:], FUTURE)
var_path = pd.DataFrame(np.cumsum(steps, axis=0) + df_us.iloc[-1].values,
                        columns=VAR_COLS,
                        index=range(int(df_us.index[-1]) + 1, int(df_us.index[-1]) + 1 + FUTURE))
var_path['Implied stock change'] = (var_path['Production'] - var_path['Domestic Consumption']
                                    - var_path['Exports'])
display(var_path.round(0))

hist = (get_series(c, 'Production') - get_series(c, 'Domestic Consumption')
        - get_series(c, 'Exports')).dropna()
print(f"historical mean implied stock change (last 20y): {hist.tail(20).mean():>10,.0f} MT/yr")
print(f"VAR  mean implied stock change                 : "
      f"{var_path['Implied stock change'].mean():>10,.0f} MT/yr")

---
## 7. Feature importance

**Permutation importance** — shuffle one feature, measure how much MAPE degrades.
Model-agnostic, and unlike tree impurity importance it does not inflate
high-cardinality features. XGBoost is the probe.

In [ ]:
def perm_importance(country, target, n_test=N_TEST, n_repeats=20):
    X, y = make_xy(country, target)
    n_test = min(n_test, len(y) - 8)
    mdl = XGBRegressor(n_estimators=300, learning_rate=.05, max_depth=3,
                       verbosity=0, random_state=RNG).fit(X.iloc[:-n_test], y.iloc[:-n_test])
    r = permutation_importance(mdl, X.iloc[-n_test:], y.iloc[-n_test:], n_repeats=n_repeats,
                               random_state=RNG, scoring='neg_mean_absolute_error')
    return pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False)


imp_us = perm_importance('United States', 'Production')
fig, ax = plt.subplots(figsize=(11, 5.5))
imp_us.head(12)[::-1].plot(kind='barh', ax=ax, color='#e91')
ax.set_title('Permutation importance — US Production (XGBoost)')
ax.set_xlabel('increase in MAE when shuffled'); plt.tight_layout(); plt.show()

In [ ]:
agg = {}
for (c, t) in all_preds:
    try:
        imp = perm_importance(c, t).clip(lower=0)
        if imp.sum() > 0:
            agg[f'{c} — {t}'] = imp / imp.sum()
    except Exception:
        pass
imp_df = pd.DataFrame(agg).fillna(0)
mean_imp = imp_df.mean(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5.5))
mean_imp.head(14)[::-1].plot(kind='barh', ax=ax, color='#57f')
ax.set_title('Mean normalised permutation importance — all series')
ax.set_xlabel('share of total importance'); plt.tight_layout(); plt.show()

grp = lambda f: 'time index' if f == 'time index' else f.split(' (t-')[0]
lag = lambda f: 'n/a' if f == 'time index' else 't-' + f.split('(t-')[1][0]
by_attr = mean_imp.groupby(grp).sum().sort_values(ascending=False)
by_lag = mean_imp.groupby(lag).sum().sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
by_attr.plot(kind='bar', ax=ax[0], color='#2a7'); ax[0].set_title('By balance-sheet attribute')
ax[0].set_ylabel('share'); ax[0].tick_params(axis='x', rotation=35)
for l in ax[0].get_xticklabels(): l.set_ha('right')
by_lag.plot(kind='bar', ax=ax[1], color='#a5c'); ax[1].set_title('By lag depth')
ax[1].set_ylabel('share'); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()
display(by_attr.round(3).to_frame('share')); display(by_lag.round(3).to_frame('share'))

### 7.1 Reading the importance results

**Production lags dominate.** Aggregated across every target series, production carries
about half the total importance — more than consumption, exports and imports combined.
This is a supply-driven market: what gets grown propagates through to everything else.

**The `time index` contributes essentially nothing.** Its importance rounds to zero,
which is the extrapolation ceiling of §5 showing up from a different angle. A tree model
cannot use a raw time index to project a trend, because every future value falls outside
the range it was split on, so the feature is useless to it. The classical models get the
trend from differencing instead.

**The lag ordering is counter-intuitive and worth flagging honestly.** `t-2` and `t-3`
carry more importance than `t-1`. A plausible reading is that the *pair* of older lags
encodes the recent slope, which matters more than the level alone for a trending series
— but this is one probe model on 20 short series, and the differences are not large
relative to the noise. It should be treated as an observation, not an established
mechanism, and re-tested on differenced targets (limitation #4) before anyone builds on
it.

---
## 8. Forecasts to 2025

Each series uses **the model that won its own backtest**, refit on the full history.

> **Selection caveat.** With 20 series and 22 models, some models win by chance. The
> per-series winner is noisy; the **median-across-series ranking in §4.1 is the
> trustworthy number**. The reported MAPE for a winning model is also optimistically
> biased, because it was selected on the same backtest it is reported on.

In [ ]:
def _xy_from_frame(df, target, n_lags=N_LAGS):
    feat = {}
    for col in df.columns:
        for L in range(1, n_lags + 1):
            feat[f'{col} (t-{L})'] = df[col].shift(L)
    feat['time index'] = df.index.to_series().values - int(df.index.min())
    X = pd.DataFrame(feat, index=df.index)
    y = df[target]
    ok = X.notna().all(axis=1) & y.notna()
    return X[ok], y[ok]


def forecast_ahead(country, target, model_name, h=FUTURE):
    y = get_series(country, target)
    idx = list(range(int(y.index[-1]) + 1, int(y.index[-1]) + 1 + h))

    if model_name == 'ARIMA':
        o = arima_order(y)
        r = ARIMA(y, order=o, trend='t' if o[1] == 0 else None).fit().get_forecast(h)
        ci = np.asarray(r.conf_int(alpha=.20))
        return pd.DataFrame({'forecast': r.predicted_mean.values,
                             'lo80': ci[:, 0], 'hi80': ci[:, 1]}, index=idx)

    if model_name == 'VAR':
        mean = [var_forecast(country, target, len(country_frame(country)), h=k + 1)
                for k in range(h)]
    elif model_name in SERIES_MODELS:
        mean = [series_forecast(model_name, y, None, h=k + 1) for k in range(h)]
    else:
        df = country_frame(country).copy()
        mean = []
        for _ in range(h):
            Xh, yh = _xy_from_frame(df, target)
            mdl = build_models()[model_name].fit(Xh, yh)
            nxt = int(df.index.max()) + 1
            df.loc[nxt] = df.loc[df.index.max()]
            Xn, _ = _xy_from_frame(df, target)
            mean.append(float(mdl.predict(Xn.iloc[[-1]])[0]))
            df.loc[nxt, target] = mean[-1]
    mean = np.asarray(mean, float)
    sd = np.std(np.diff(y.values), ddof=1)
    band = 1.2816 * sd * np.sqrt(np.arange(1, h + 1))
    return pd.DataFrame({'forecast': mean, 'lo80': mean - band, 'hi80': mean + band}, index=idx)


best_by_series = scores.loc[scores.groupby(['country', 'series']).MAPE.idxmin()]
forecasts = {}
for _, r in best_by_series.iterrows():
    try:
        forecasts[(r.country, r.series)] = forecast_ahead(r.country, r.series, r.model).clip(lower=0)
    except Exception as e:
        print('failed', r.country, r.series, r.model, type(e).__name__)
print(f'{len(forecasts)} forecasts for market years 2021-2025')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, t in zip(axes.ravel(), TARGETS):
    key = ('United States', t)
    if key not in forecasts:
        ax.axis('off'); continue
    y, fc = get_series(*key), forecasts[key]
    mn = best_by_series.query('country == "United States" and series == @t').model.iloc[0]
    ax.plot(y.index, y.values / 1000, color='#333', lw=1.4, label='observed')
    ax.plot(fc.index, fc.forecast / 1000, color='#c33', lw=2, marker='o', ms=4,
            label=f'forecast ({mn})')
    ax.fill_between(fc.index, fc.lo80 / 1000, fc.hi80 / 1000, color='#c33', alpha=.18,
                    label='80% interval')
    ax.set_title(f'United States — {t}'); ax.set_ylabel('1000 MT')
    ax.set_xlabel('market year'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
regions = sorted({c for c, _ in forecasts})
fig, axes = plt.subplots(len(regions), 2, figsize=(14, 3.1 * len(regions)), squeeze=False)
for i, c in enumerate(regions):
    for j, t in enumerate(['Production', 'Domestic Consumption']):
        ax = axes[i, j]
        if (c, t) not in forecasts:
            ax.axis('off'); continue
        y, fc = get_series(c, t), forecasts[(c, t)]
        ax.plot(y.index, y.values / 1000, color='#333', lw=1.2)
        ax.plot(fc.index, fc.forecast / 1000, color='#c33', lw=1.8, marker='o', ms=3)
        ax.fill_between(fc.index, fc.lo80 / 1000, fc.hi80 / 1000, color='#c33', alpha=.18)
        ax.set_title(f'{c} — {t}', fontsize=10); ax.set_ylabel('1000 MT')
plt.tight_layout(); plt.show()

In [ ]:
summary = []
for (c, t), fc in forecasts.items():
    y = get_series(c, t)
    r = best_by_series.query('country == @c and series == @t').iloc[0]
    # Spain/Italy end at MY2000, everyone else at MY2020 -- so the "actual" column is
    # labelled generically and the year it refers to carried alongside it.
    summary.append({'country': c, 'series': t, 'model': r.model,
                    'backtest MAPE %': r.MAPE,
                    'last obs year': int(y.index[-1]), 'actual (last obs)': y.iloc[-1],
                    'forecast year': int(fc.index[-1]), 'forecast (+5y)': fc.forecast.iloc[-1],
                    'lo80': fc.lo80.iloc[-1], 'hi80': fc.hi80.iloc[-1],
                    'change %': (fc.forecast.iloc[-1] / y.iloc[-1] - 1) * 100 if y.iloc[-1] else np.nan})
summary = pd.DataFrame(summary).sort_values(['country', 'series'])
display(summary.round(1))
print('Note: Spain and Italy end at MY2000 (EU aggregation), so their +5y horizon is '
      'MY2005 — historical backcast, not a 2025 projection.')

In [ ]:
growth = {}
for (c, t), fc in forecasts.items():
    if t != 'Production':
        continue
    y = get_series(c, t)
    # Only regions whose history actually reaches MY2020 belong on a 2020->2025 map.
    # Spain/Italy stop at MY2000, so their +5y horizon is 2005 and would be misleading.
    if int(y.index[-1]) == 2020 and y.iloc[-1]:
        growth[c] = (fc.forecast.iloc[-1] / y.iloc[-1] - 1) * 100
g = to_map_frame(pd.Series(growth), 'growth'); lim = max(1.0, g.growth.abs().max())
fig = px.choropleth(g, locations='iso', color='growth', hover_name='label',
                    color_continuous_scale='RdYlGn', range_color=(-lim, lim),
                    title='Forecast change in almond production, 2020 → 2025 (%)')
fig.update_layout(height=450, margin=dict(l=0, r=0, t=50, b=0)); fig.show()
pd.Series(growth).round(1).to_frame('forecast growth 2020→2025 (%)')

### 8.1 Do the independent forecasts balance?

In [ ]:
us_p = forecasts[('United States', 'Production')].forecast
us_c = forecasts[('United States', 'Domestic Consumption')].forecast
us_e = forecasts[('United States', 'Exports')].forecast
balance = pd.DataFrame({'Production': us_p, 'Domestic Consumption': us_c, 'Exports': us_e,
                        'Implied stock change': us_p - us_c - us_e})
display(balance.round(0))
print(f'historical mean implied stock change (last 20y): {hist.tail(20).mean():>10,.0f} MT/yr')
print(f'independent forecasts, mean implied            : {balance["Implied stock change"].mean():>10,.0f} MT/yr')
print(f'VAR system forecast, mean implied              : {var_path["Implied stock change"].mean():>10,.0f} MT/yr')

---
## 9. Findings, limitations, and proposed fixes

### Findings — data

1. `Market_Year` is the PSD time axis. `Calendar_Year`/`Month` are publication metadata
   and collapse the data to **17 usable timestamps** with 1,992 rows lost to `NaT`.
2. The data is **annual**, which rules out Seasonal Naive, SARIMA's seasonal terms,
   `seasonal_decompose`, and most of Prophet's value proposition.
3. Spain and Italy become **all-zero placeholders** from MY2001 when they merge into the
   `European Union` line. They must be masked, not read as zeros.
4. Both PSD accounting identities hold **to the tonne**, which validates the reshape.

### Findings — demand and supply

5. **Supply is extraordinarily concentrated** — the US is ~80% of world production and
   its share has *grown*. World almond supply is effectively a bet on California.
6. **Demand is dispersed** — the EU, India, China and the Middle East are the large
   consumers. The EU is structurally a net importer.
7. Production, consumption and exports trend strongly upward. **Imports are the noisiest
   and least predictable component in every region.**

### Findings — modelling

8. **Simple methods win.** Across 22 models under rolling-origin validation, the
   baseline and classical tiers take the majority of series. See §4.1 for the ranking.
9. **Tree and boosting models cannot extrapolate.** A Random Forest emits a flat line
   for eight consecutive years — its output is bounded by the training maximum — while
   actual production rose ~48%. This is a property of the model class, not a tuning
   failure, and no hyperparameter search fixes it.
10. **Unregularised LinearRegression is dangerous here**, not merely mediocre: it lands
    near the bottom of the table with a worst-case several times its median. With 19
    correlated lag features and ~58 rows it is badly unstable. **Ridge and Lasso fix
    most of this with nothing but a penalty term** — if you are using plain linear
    regression on lag features anywhere, regularise it.
11. **Architecture beats depth for the neural models.** The LSTM and GRU clearly
    outperform the plain MLPs on identical data and features, despite being *smaller*
    models. An MLP sees the lag features as an unordered vector and must learn temporal
    ordering from data; a recurrent net receives that ordering as an architectural
    prior. When data is scarce, a good inductive bias substitutes for data. The deeper
    MLP is consistently worse than the shallower one — capacity was never the
    constraint.
12. **But the recurrent nets still lose to Drift and ETS.** Better than the other ML
    models is not the same as better than a well-chosen simple model.
13. **VAR loses badly on accuracy but wins decisively on coherence.** It ranks near the
    bottom for per-series MAPE — heavily over-parameterised at roughly 36 coefficients
    across a four-variable, two-lag system fitted on as few as 17 observations. But on
    the balance check it is **~3.4× closer to the historical norm** than independent
    forecasting (§6, §8.1). Accuracy and coherence are different objectives here, and
    the model that wins one loses the other. If the joint picture matters — and for a
    supply/demand study it does — VAR earns its place despite the MAPE.
14. **Production lags dominate feature importance**, carrying about half the total
    across all target series: this is a supply-driven market. The `time index` feature
    contributes essentially **zero**, which is the extrapolation ceiling appearing from
    another angle — a tree cannot use a raw time index to project beyond its training
    range.
15. **The lag ordering is counter-intuitive:** `t-2` and `t-3` outrank `t-1`. Plausibly
    the older pair encodes recent slope, but this is one probe model on 20 short series
    and the gaps are not large relative to noise. Treat it as an observation to re-test
    (limitation #4), not an established mechanism.

---

## Predictions

The §8 summary table gives per-series point forecasts to MY2025 with 80% intervals.
Read them with three caveats:

- **Intervals are wide, and that is the honest signal.** Quote them alongside the point
  estimates, never on their own.
- **The winner-per-series is noisy.** With 20 series and 22 models some models win by
  chance. The median-across-series ranking in §4.1 is the trustworthy comparison, and
  the winning model's own MAPE is optimistically biased because it was selected on the
  backtest it is reported on.
- **The production forecasts assume the boom continues.** Extrapolating the post-2015
  California acreage surge linearly to 2025 is an aggressive assumption, and it is the
  main reason the balance check below fails.

---

## Limitations and proposed fixes

| # | Limitation | Why it matters | Proposed fix |
|---|---|---|---|
| 1 | **Independent forecasts don't satisfy the balance identity** | Implied US stock build runs far above any historical value, so the joint picture is incoherent even where individual series look fine | Forecast three components and **derive the fourth from the identity**; or apply **hierarchical reconciliation** (MinT/OLS) to project independent forecasts onto the constraint. VAR is the textbook answer but is over-parameterised here (§6) — a **Bayesian VAR with Minnesota priors** would shrink it enough to be usable |
| 2 | **Very short series** — 20–61 annual points | Rules out deep learning at full strength, makes every parameter estimate noisy, and makes model selection itself unstable | Pool countries into a **global/panel model** — one model across all series with country features — which multiplies effective sample size and is the standard fix for exactly this. Or extend history with **FAOSTAT** and national statistics |
| 3 | **Annual granularity** | No seasonality, no within-year dynamics, only ~10 usable validation points per series | Bring in **monthly trade data** (UN Comtrade, USDA export sales) for the trade components, which supports genuinely seasonal models |
| 4 | **ML models were fed levels, not differences** | Guarantees the extrapolation ceiling in §5 — the tree models were structurally unable to win | **Re-run the bake-off on differenced or log-growth targets** and cumulate back. This is the single highest-value remaining experiment and the fairest test of whether ML can compete |
| 5 | **No exogenous drivers** | Almond output is driven by bearing acreage, California water allocations, drought, frost and tariffs — none of which are in the model | Add **ARIMAX/SARIMAX regressors**: USDA/NASS bearing acreage, PDSI drought index, water allocation percentages, and tariff dummies (notably the 2018 India/China episode) |
| 6 | **Structural break at MY2001 handled only by masking** | Other regime changes may lurk undetected | Run **Chow / Bai-Perron break tests** per series, then segment the sample or add break dummies |
| 7 | **Approximate forecast intervals** | Non-ARIMA intervals use a normal approximation from in-sample first differences, which understates tail risk | Use **conformal prediction** or a block bootstrap for distribution-free intervals with real coverage guarantees |
| 8 | **Interpolated missing years** | 1963 (all) and 1977–1980 (US) are linearly filled, understating true uncertainty | Treat them as genuinely missing via a **state-space / Kalman** formulation, which handles gaps natively instead of inventing values |
| 9 | **Winner selected on the same backtest it is reported on** | The headline MAPE for each winning model is optimistically biased | Add a **nested/outer holdout** — select on an inner split, report on a final untouched period |
| 10 | **Single random seed** | Neural and tree results shift with initialisation; some ranking differences are noise | Repeat across **multiple seeds** and report mean ± sd, rather than a single run |
| 11 | **Hyperparameters largely untuned** | The ML tier was given sensible defaults, not a search; its ranking is a floor, not a ceiling | Nested **time-series CV** (`TimeSeriesSplit`) for hyperparameter search inside each training window |
| 12 | **Forecasts assume no supply shock** | A drought, frost or tariff event breaks every model here simultaneously | Present **scenarios** (boom / plateau / drought-shock) rather than one path, and cap growth using known bearing-acreage plantings, which are observable years in advance |